# 🐟 Fish-Speech LoRA Fine-Tuning: Bahasa Indonesia (T4 16GB GPU)

Notebook ini melatih model TTS multilingual **Fish-Speech 1.5** menggunakan dataset publik **`agufsamudra/tts-indo`** (4.531 sampel audio narasi Bahasa Indonesia).

### ⚡ Persyaratan:
- Runtime: **GPU (T4 16GB VRAM - Gratis)** di Google Colab.
- Pastikan menu: **Runtime > Change runtime type > T4 GPU** sudah terpilih.

In [ ]:
# 1. Verifikasi Akses GPU Colab (T4 16GB VRAM)
!nvidia-smi

In [ ]:
# 2. Clone Fish-Speech Repository (Tag Stabil v1.5.1) & Install Dependensi
import os
from pathlib import Path

# Pastikan selalu berada di direktori root /content
%cd /content

# Clone fish-speech release v1.5.1 jika belum ada
if not os.path.exists('/content/fish-speech'):
    !git clone --branch v1.5.1 --single-branch https://github.com/fishaudio/fish-speech.git /content/fish-speech

%cd /content/fish-speech

# 1. Pasang paket sistem audio Linux & build tools
!apt-get update -qq && apt-get install -y -qq libasound2-dev libsox-dev portaudio19-dev

# 2. Install fish-speech tanpa build desktop wheels yang bermasalah di cloud
!pip install -q --no-deps -e .

# 3. Install seluruh paket dependensi training yang telah diverifikasi
!pip install -q 'protobuf>=3.20.0,<6.0.0' vector_quantize_pytorch 'einx[torch]' lightning hydra-core einops pyrootutils natsort tiktoken loguru loralib transformers datasets soundfile scipy pyarrow pydub huggingface_hub

# 4. Patch kompatibilitas torchaudio 2.x langsung dari Python (tanpa script eksternal)
f_vq = Path('tools/vqgan/extract_vq.py')
if f_vq.exists():
    c = f_vq.read_text(encoding='utf-8')
    if 'try:\n    backends = torchaudio.list_audio_backends()' not in c:
        c = c.replace(
            'backends = torchaudio.list_audio_backends()',
            'try:\n    backends = torchaudio.list_audio_backends()\nexcept Exception:\n    backends = ["soundfile"]'
        )
        f_vq.write_text(c, encoding='utf-8')
        print('Patched tools/vqgan/extract_vq.py')

f_ref = Path('fish_speech/inference_engine/reference_loader.py')
if f_ref.exists():
    c = f_ref.read_text(encoding='utf-8')
    if 'try:\n            backends = torchaudio.list_audio_backends()' not in c:
        c = c.replace(
            '        backends = torchaudio.list_audio_backends()',
            '        try:\n            backends = torchaudio.list_audio_backends()\n        except Exception:\n            backends = ["soundfile"]'
        )
        f_ref.write_text(c, encoding='utf-8')
        print('Patched fish_speech/inference_engine/reference_loader.py')

print('Environment & compatibility patch berhasil!')


In [ ]:
# 3. Download Base Model Checkpoint (fish-speech-1.5) dari HuggingFace
# Model ini bersifat publik dan tidak memerlukan approval lisensi khusus
from huggingface_hub import snapshot_download
import os

os.makedirs('/content/fish-speech/checkpoints/fish-speech-1.5', exist_ok=True)

snapshot_download(
    repo_id='fishaudio/fish-speech-1.5',
    local_dir='/content/fish-speech/checkpoints/fish-speech-1.5',
    ignore_patterns=['*.bin', '*.msgpack', '*.safetensors.index.json']
)
print('Pretrained checkpoint fish-speech-1.5 berhasil diunduh!')


In [ ]:
# 4. Download & Preprocess Dataset Indonesia (agufsamudra/tts-indo)
import os
import io
import soundfile as sf
import numpy as np
from scipy import signal
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
import pyarrow.parquet as pq

# Reset path ke direktori utama repo
%cd /content/fish-speech

output_dir = Path('data/Speaker_Indonesia')
output_dir.mkdir(parents=True, exist_ok=True)

api = HfApi()
dataset_repo = 'agufsamudra/tts-indo'
files = [f for f in api.list_repo_files(dataset_repo, repo_type='dataset') if f.startswith('data/train-') and f.endswith('.parquet')]

TARGET_SR = 24000
MAX_SAMPLES = 1000  # Default 1000 sampel untuk fine-tuning di Colab T4/A100. Ubah ke None untuk seluruh 114.036 file.
count = 0

print(f'Mengunduh dan mengekstrak audio dari {len(files)} partisi dataset...')
for pq_file in files:
    if MAX_SAMPLES and count >= MAX_SAMPLES:
        break
    p_local = hf_hub_download(dataset_repo, pq_file, repo_type='dataset')
    table = pq.read_table(p_local)
    for i in range(table.num_rows):
        if MAX_SAMPLES and count >= MAX_SAMPLES:
            break
        row = {c: table[c][i].as_py() for c in table.column_names}
        text = str(row.get('text', '')).strip()
        audio_obj = row.get('audio')
        if not text or not audio_obj:
            continue
        b = audio_obj['bytes'] if isinstance(audio_obj, dict) else audio_obj
        data, sr = sf.read(io.BytesIO(b), dtype='float32')
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        if sr != TARGET_SR:
            data = signal.resample(data, int(len(data) * TARGET_SR / sr))
        # Normalisasi amplitudo
        mx = np.max(np.abs(data))
        if mx > 1e-5:
            data = data / mx * 0.90
        count += 1
        wav_name = f'segment_{count:05d}.wav'
        lab_name = f'segment_{count:05d}.lab'
        sf.write(str(output_dir / wav_name), data.astype(np.float32), TARGET_SR, format='WAV', subtype='PCM_16')
        with open(output_dir / lab_name, 'w', encoding='utf-8') as f:
            f.write(text)
        if count % 50 == 0:
            print(f'Progress: {count} file audio .wav dan .lab disiapkan...')

print(f'Total {count} sampel audio 24kHz dan label teks berhasil disiapkan di data/Speaker_Indonesia!')


In [ ]:
# 5. Ekstraksi Semantic VQ Tokens (Firefly-GAN VQ Codec)
%cd /content/fish-speech

!python tools/vqgan/extract_vq.py 'data/Speaker_Indonesia' \
    --num-workers 2 \
    --batch-size 16 \
    --config-name 'firefly_gan_vq' \
    --checkpoint-path 'checkpoints/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth'

print('Semantic VQ extraction selesai! File .npy token berhasil digenerasi.')


In [ ]:
# 6. Build Protobuf Dataset untuk Dual-AR LLaMA Training
%cd /content/fish-speech

!python tools/llama/build_dataset.py \
    --input 'data/Speaker_Indonesia' \
    --output 'data/protos' \
    --text-extension .lab \
    --num-workers 2

print('Protobuf dataset siap untuk training di data/protos!')


In [ ]:
# 7. Jalankan Dual-AR LoRA Fine-Tuning pada T4 GPU
%cd /content/fish-speech

!python fish_speech/train.py \
    --config-name 'text2semantic_finetune' \
    project='indonesia-tts' \
    +lora@model.model.lora_config=r_8_alpha_16 \
    trainer.max_steps=200 \
    trainer.val_check_interval=50 \
    trainer.strategy=auto \
    data.num_workers=2

print('Training LoRA selesai! Checkpoint tersimpan di results/indonesia-tts/checkpoints/')


In [ ]:
# 8. Merge LoRA Checkpoint ke Base Model & Test Sintesis Suara Bahasa Indonesia
%cd /content/fish-speech
import glob
import os
import torch
import soundfile as sf
from pathlib import Path
from IPython.display import Audio, display

# Cari file checkpoint LoRA terakhir
ckpts = sorted(glob.glob('results/indonesia-tts/checkpoints/*.ckpt'))
if not ckpts:
    raise FileNotFoundError('Tidak ada checkpoint .ckpt yang ditemukan di results/indonesia-tts/checkpoints/')

latest_ckpt = ckpts[-1]
print(f'Menggabungkan LoRA checkpoint: {latest_ckpt}')

# Gabungkan bobot LoRA ke bobot model dasar
!python tools/llama/merge_lora.py \
    --lora-config r_8_alpha_16 \
    --base-weight checkpoints/fish-speech-1.5 \
    --lora-weight {latest_ckpt} \
    --output checkpoints/indonesia-tts-merged

print('Bobot LoRA berhasil digabungkan ke checkpoints/indonesia-tts-merged!')

# Uji coba sintesis suara Bahasa Indonesia langsung di Colab
from fish_speech.inference_engine import TTSInferenceEngine
from fish_speech.models.text2semantic.inference import launch_thread_safe_queue
from fish_speech.models.vqgan.inference import load_model as load_decoder_model
from fish_speech.utils.schema import ServeTTSRequest

llama_queue = launch_thread_safe_queue(
    checkpoint_path='checkpoints/indonesia-tts-merged',
    device='cuda',
    precision=torch.bfloat16,
    compile=False,
)
decoder_model = load_decoder_model(
    config_name='firefly_gan_vq',
    checkpoint_path='checkpoints/fish-speech-1.5/firefly-gan-vq-fsq-8x1024-21hz-generator.pth',
    device='cuda',
)
engine = TTSInferenceEngine(
    llama_queue=llama_queue,
    decoder_model=decoder_model,
    precision=torch.bfloat16,
    compile=False,
)

test_sentence = 'Halo semua! Selamat datang di episode terbaru podcast kita. Hari ini kita membahas kecerdasan buatan.'
print(f"Synthesizing audio: '{test_sentence}'...")

results = list(engine.inference(
    ServeTTSRequest(
        text=test_sentence,
        references=[],
        max_new_tokens=256,
        chunk_length=150,
        top_p=0.7,
        repetition_penalty=1.2,
        temperature=0.7,
        format='wav',
    )
))

out_file = 'test_indonesia.wav'
for r in results:
    if r.code == 'final' and r.audio is not None:
        sr, audio_arr = r.audio
        sf.write(out_file, audio_arr, sr)
        print(f'Sintesis berhasil! Audio tersimpan di {out_file}')

display(Audio(out_file))


In [ ]:
# 9. Download Checkpoint LoRA ke PC Lokal Anda
%cd /content/fish-speech

# Kompres LoRA checkpoint untuk diunduh
!zip -r /content/indonesia-tts-lora.zip results/indonesia-tts/checkpoints/

from google.colab import files
print('Mengunduh indonesia-tts-lora.zip ke browser Anda...')
files.download('/content/indonesia-tts-lora.zip')
print('File checkpoint siap dipindahkan ke folder services/fish-speech/checkpoints di PC lokal Anda!')
